# Ariel data challenge submission

## 1. Notebook set up

In [1]:
# Standard library imports
import os
import pickle
import time

# Third party imports
import numpy as np
import pandas as pd
import tensorflow as tf

# Project imports
from ariel_data_preprocessing.data_preprocessing import DataProcessor

mode = 'submission'

if mode == 'testing':
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    INPUT_DIRECTORY = 'data/raw'
    OUTPUT_DIRECTORY = 'data/processed'
    REBUILD_DATA = True
    N_PLANETS = 10
    MODEL = 'data/models/ariel-cnn-8.1M-2ksteps-tf2.11.keras'

elif mode == 'submission':
    INPUT_DIRECTORY = '/kaggle/input/ariel-data-challenge-2025'
    OUTPUT_DIRECTORY = '/kaggle/working'
    REBUILD_DATA = True
    N_PLANETS = -1
    MODEL = '/kaggle/input/ariel-cnn/ariel-cnn-8.1M-862ksteps.keras'

2025-09-24 20:32:30.286599: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758745950.564898      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758745950.644726      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## 2. Data preparation

### 2.1. Preprocess the raw data

In [2]:
data_processor = DataProcessor(
    input_data_path=INPUT_DIRECTORY,
    output_data_path=OUTPUT_DIRECTORY,
    output_filename='test.h5',
    n_cpus=4,
    downsample_fgs=True,
    n_planets=N_PLANETS,
    mode='test'
)

In [3]:
if REBUILD_DATA:
    start_time = time.time()
    data_processor.run()
    end_time = time.time()

    print(f'Data preprocessing completed in {(end_time - start_time)/60:.2f} minutes')
    print(f'Total test planets: {len(data_processor.planet_list)}')

Data preprocessing completed in 0.39 minutes
Total test planets: 1


### 2.2. Initialize data generator

In [4]:
data_processor.initialize_data_generators(
    sample_size=372,
    n_samples=10
)

2025-09-24 20:33:14.314735: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### 2.3. Create dataset

In [5]:
testing_data = data_processor.testing.take(len(data_processor.planet_list))
signals = np.array([element.numpy() for element in testing_data])

print(f'Signals shape: {signals.shape}')

Signals shape: (1, 10, 372, 283)


## 3. Predictions

In [6]:
model = tf.keras.models.load_model(MODEL)

spectrum_predictions = []

for planet in signals:
    spectrum_predictions.append(model.predict(planet, batch_size=10, verbose=0))

spectrum_predictions = np.array(spectrum_predictions)
spectrum_predictions_avg = np.mean(spectrum_predictions, axis=1)
spectrum_predictions_std = np.std(spectrum_predictions, axis=1)

print(f'Spectrum predictions shape: {spectrum_predictions.shape}')
print(f'Spectrum predictions avg shape: {spectrum_predictions_avg.shape}')
print(f'Spectrum predictions std shape: {spectrum_predictions_std.shape}')

I0000 00:00:1758745997.689422     107 service.cc:148] XLA service 0x79c8dc007c80 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758745997.690395     107 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1758745998.287280     107 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Spectrum predictions shape: (1, 10, 283)
Spectrum predictions avg shape: (1, 283)
Spectrum predictions std shape: (1, 283)


## 4. Error correction

In [7]:
scaled_errors = spectrum_predictions_std * 323

print(f'Raw errors: {spectrum_predictions_std[0][:5]}')
print(f'Corrected errors: {scaled_errors[0][:5]}')
print(f'Corrected errors shape: {scaled_errors.shape}')

Raw errors: [0.00012342 0.00012617 0.00012108 0.00012052 0.00012311]
Corrected errors: [0.03986548 0.04075304 0.03910917 0.03892782 0.03976405]
Corrected errors shape: (1, 283)


## 5. Submission file

In [8]:
submission = np.concatenate(
    (spectrum_predictions_avg, scaled_errors),
    axis=1
)

submission_df = pd.DataFrame(submission)

col_names = [f'wl_{i}' for i in range(1, 284)]
col_names += [f'sigma_{i}' for i in range(1, 284)]
submission_df.columns = col_names

submission_df.insert(0, 'planet_id', data_processor.planet_list)
submission_df['planet_id'] = submission_df['planet_id'].astype(int)

submission_df.to_csv('submission.csv', index=False)
submission_df.head()

,planet_id,wl_1,wl_2,wl_3,wl_4,wl_5,wl_6,wl_7,wl_8,wl_9,...,sigma_274,sigma_275,sigma_276,sigma_277,sigma_278,sigma_279,sigma_280,sigma_281,sigma_282,sigma_283
0,1103775,0.018372,0.017677,0.018354,0.018728,0.017916,0.018321,0.018675,0.018151,0.018488,...,0.039117,0.039573,0.039233,0.039472,0.039178,0.040272,0.039452,0.039362,0.041049,0.039378
